# Quantitative Value Strategy
"Value investing" means investing in the stocks that are cheapest relative to common measures of business value (like earnings or assets).

For this project, we're going to build an investing strategy that selects the 50 stocks with the best value metrics. From there, we will calculate recommended trades for an equal-weight portfolio of these 50 stocks.

## Library Imports
The first thing we need to do is import the open-source software libraries that we'll be using in this tutorial.

In [1]:
import numpy as np
import pandas as pd
import xlsxwriter
import requests
from scipy import stats
import math
import yfinance as yf

## Importing Our List of Stocks & API Token
As before, we'll need to import our list of stocks and our API token before proceeding. Make sure the .csv file is still in your working directory and import it with the following command:

In [2]:
stocks = pd.read_csv('sp_500_stocks.csv')

## Making Our First API Call
It's now time to make the first version of our value screener!

We'll start by building a simple value screener that ranks securities based on a single metric (the price-to-earnings ratio).

In [3]:
symbol = 'AAPL'
ticker = yf.Ticker(symbol)
data = ticker.info
data

{'address1': 'One Apple Park Way',
 'city': 'Cupertino',
 'state': 'CA',
 'zip': '95014',
 'country': 'United States',
 'phone': '(408) 996-1010',
 'website': 'https://www.apple.com',
 'industry': 'Consumer Electronics',
 'industryKey': 'consumer-electronics',
 'industryDisp': 'Consumer Electronics',
 'sector': 'Technology',
 'sectorKey': 'technology',
 'sectorDisp': 'Technology',
 'longBusinessSummary': 'Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide. The company offers iPhone, a line of smartphones; Mac, a line of personal computers; iPad, a line of multi-purpose tablets; and wearables, home, and accessories comprising AirPods, Apple Vision Pro, Apple TV, Apple Watch, Beats products, and HomePod, as well as Apple branded and third-party accessories. It also provides AppleCare support and cloud services; and operates various platforms, including the App Store that allow customers to discover and download app

## Parsing Our API Call
This API call has the metric we need - the price-to-earnings ratio.

Here is an example of how to parse the metric from our API call:

In [4]:
price = data['currentPrice']
pe_ratio = data['trailingPE']

## Executing A Batch API Call & Building Our DataFrame

Just like in our first project, it's now time to execute several batch API calls and add the information we need to our DataFrame.

We'll start by running the following code cell, which contains some code we already built last time that we can re-use for this project. More specifically, it contains a function called chunks that we can use to divide our list of securities into groups of 100.

In [5]:
# Function sourced from 
# https://stackoverflow.com/questions/312443/how-do-you-split-a-list-into-evenly-sized-chunks
def chunks(lst, n):
    """Yield successive n-sized chunks from lst."""
    for i in range(0, len(lst), n):
        yield lst[i:i + n]   
        
symbol_groups = list(chunks(stocks['Ticker'], 100))
symbol_strings = []
for i in range(0, len(symbol_groups)):
    symbol_strings.append(','.join(symbol_groups[i]))
#     print(symbol_strings[i])

my_columns = ['Ticker', 'Price', 'Price-to-Earnings Ratio', 'Number of Shares to Buy']

Now we need to create a blank DataFrame and add our data to the data frame one-by-one.

In [8]:
final_dataframe = pd.DataFrame(columns=my_columns)

for symbol in stocks['Ticker']:
    try:
        ticker = yf.Ticker(symbol)
        data = ticker.info
        final_dataframe = pd.concat([final_dataframe, pd.Series([symbol,
                                                                  data.get('currentPrice') or data.get('regularMarketPrice'),
                                                                  data.get('trailingPE'),
                                                                  'N/A'
                                                                  ],
                                                                 index=my_columns).to_frame().T],
                                    ignore_index=True)
    except Exception as e:
        print(f"Skipping {symbol}: {e}")

final_dataframe

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ANSS"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CMA"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: COG"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: FRC"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: HBI"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: HES"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: IPG"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol:

,Ticker,Price,Price-to-Earnings Ratio,Number of Shares to Buy
0,A,121.11,26.735098,N/A
1,AAL,13.345,78.5,N/A
2,AAP,58.15,51.46018,N/A
3,AAPL,269.945,34.170254,N/A
4,ABBV,211.5,89.618645,N/A
...,...,...,...,...
500,YUM,162.36,29.254053,N/A
501,ZBH,96.31,27.129578,N/A
502,ZBRA,239.0,29.217604,N/A
503,ZION,62.93,10.470881,N/A


## Removing Glamour Stocks

The opposite of a "value stock" is a "glamour stock". 

Since the goal of this strategy is to identify the 50 best value stocks from our universe, our next step is to remove glamour stocks from the DataFrame.

We'll sort the DataFrame by the stocks' price-to-earnings ratio, and drop all stocks outside the top 50.

In [19]:
final_dataframe.sort_values('Price-to-Earnings Ratio', inplace = True)
final_dataframe = final_dataframe[final_dataframe['Price-to-Earnings Ratio'] > 0]
final_dataframe = final_dataframe[:50]
final_dataframe.reset_index(inplace = True)
final_dataframe.drop('index', axis=1, inplace = True)
final_dataframe


,Ticker,Price,Price-to-Earnings Ratio,Number of Shares to Buy
0,CMCSA,29.715,5.512987,N/A
1,ALL,216.02,5.674284,N/A
2,DXC,13.205,5.741304,N/A
3,EIX,70.12,6.070996,N/A
4,WU,9.415,6.194079,N/A
5,KSS,14.76,6.201681,N/A
6,LNC,36.93,6.334477,N/A
7,CHTR,239.17,6.603258,N/A
8,VNO,29.24,6.961905,N/A
9,LEG,12.195,7.215976,N/A


## Calculating the Number of Shares to Buy
We now need to calculate the number of shares we need to buy. 

To do this, we will use the `portfolio_input` function that we created in our momentum project.

I have included this function below.

In [20]:
def portfolio_input():
    global portfolio_size
    portfolio_size = input("Enter the value of your portfolio:")

    try:
        val = float(portfolio_size)
    except ValueError:
        print("That's not a number! \n Try again:")
        portfolio_size = input("Enter the value of your portfolio:")

Use the `portfolio_input` function to accept a `portfolio_size` variable from the user of this script.

In [21]:
portfolio_input()

You can now use the global `portfolio_size` variable to calculate the number of shares that our strategy should purchase.

In [24]:
position_size = float(portfolio_size) / len(final_dataframe.index)

for i in range(0, len(final_dataframe['Ticker'])):
    final_dataframe.loc[i, 'Number of Shares to Buy'] = math.floor(position_size / final_dataframe['Price'][i])
final_dataframe

,Ticker,Price,Price-to-Earnings Ratio,Number of Shares to Buy
0,CMCSA,29.715,5.512987,673
1,ALL,216.02,5.674284,92
2,DXC,13.205,5.741304,1514
3,EIX,70.12,6.070996,285
4,WU,9.415,6.194079,2124
5,KSS,14.76,6.201681,1355
6,LNC,36.93,6.334477,541
7,CHTR,239.17,6.603258,83
8,VNO,29.24,6.961905,683
9,LEG,12.195,7.215976,1640


## Building a Better (and More Realistic) Value Strategy
Every valuation metric has certain flaws.

For example, the price-to-earnings ratio doesn't work well with stocks with negative earnings.

Similarly, stocks that buyback their own shares are difficult to value using the price-to-book ratio.

Investors typically use a `composite` basket of valuation metrics to build robust quantitative value strategies. In this section, we will filter for stocks with the lowest percentiles on the following metrics:

* Price-to-earnings ratio
* Price-to-book ratio
* Price-to-sales ratio
* Enterprise Value divided by Earnings Before Interest, Taxes, Depreciation, and Amortization (EV/EBITDA)
* Enterprise Value divided by Gross Profit (EV/GP)

Some of these metrics aren't provided directly by the IEX Cloud API, and must be computed after pulling raw data. We'll start by calculating each data point from scratch.

In [25]:
symbol = 'AAPL'
ticker = yf.Ticker(symbol)
data = ticker.info

# P/E Ratio
pe_ratio = data['trailingPE']
# P/B Ratio
pb_ratio = data['priceToBook']
# P/S Ratio
ps_ratio = data['priceToSalesTrailing12Months']
# EV/EBITDA
enterprise_value = data['enterpriseValue']
ebitda = data['ebitda']
ev_to_ebitda = enterprise_value / ebitda
# EV/GP
gross_profit = data['grossProfits']
ev_to_gross_profit = enterprise_value / gross_profit

Let's move on to building our DataFrame. You'll notice that I use the abbreviation `rv` often. It stands for `robust value`, which is what we'll call this sophisticated strategy moving forward.

In [26]:
rv_columns = [
    'Ticker',
    'Price',
    'Number of Shares to Buy',
    'Price-to-Earnings Ratio',
    'PE Percentile',
    'Price-to-Book Ratio',
    'PB Percentile',
    'Price-to-Sales Ratio',
    'PS Percentile',
    'EV/EBITDA',
    'EV/EBITDA Percentile',
    'EV/GP',
    'EV/GP Percentile',
    'RV Score'
]

rv_dataframe = pd.DataFrame(columns=rv_columns)

for symbol in stocks['Ticker']:
    try:
        ticker = yf.Ticker(symbol)
        data = ticker.info

        enterprise_value = data.get('enterpriseValue')
        ebitda = data.get('ebitda')
        gross_profit = data.get('grossProfits')

        try:
            ev_to_ebitda = enterprise_value / ebitda
        except TypeError:
            ev_to_ebitda = np.NaN

        try:
            ev_to_gross_profit = enterprise_value / gross_profit
        except TypeError:
            ev_to_gross_profit = np.NaN

        rv_dataframe = pd.concat([rv_dataframe, pd.Series([
                symbol,
                data.get('currentPrice') or data.get('regularMarketPrice'),
                'N/A',
                data.get('trailingPE'),
                'N/A',
                data.get('priceToBook'),
                'N/A',
                data.get('priceToSalesTrailing12Months'),
                'N/A',
                ev_to_ebitda,
                'N/A',
                ev_to_gross_profit,
                'N/A',
                'N/A'
        ], index=rv_columns).to_frame().T], ignore_index=True)

    except Exception as e:
        print(f"Skipping {symbol}: {e}")

rv_dataframe

Skipping ABC: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping ABMD: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping ABT: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping ALXN: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping AMP: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ANSS"}}}


Skipping ANSS: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping ANTM: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping ATVI: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping AXP: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping BAC: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping BF.B: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping BK: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping BLL: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping BRK.B: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping C: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping CERN: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping CFG: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CMA"}}}


Skipping CMA: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping COF: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping COG: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping CTL: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping CTXS: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping CXO: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping DFS: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping DISCA: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping DISCK: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping DISH: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping DRE: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping ETFC: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instea

...

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: K"}}}


Skipping K: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping KEY: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping KSU: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping MMC: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping MRO: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping MS: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping MTB: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping MXIM: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping MYL: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping NBL: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping NLOK: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping NLSN: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skip

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: WBA"}}}


Skipping WBA: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping WFC: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping WLTW: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping WRK: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping XLNX: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.
Skipping ZION: `np.NaN` was removed in the NumPy 2.0 release. Use `np.nan` instead.


,Ticker,Price,Number of Shares to Buy,Price-to-Earnings Ratio,PE Percentile,Price-to-Book Ratio,PB Percentile,Price-to-Sales Ratio,PS Percentile,EV/EBITDA,EV/EBITDA Percentile,EV/GP,EV/GP Percentile,RV Score
0,A,121.17,N/A,26.748343,N/A,4.958667,N/A,None,N/A,17.863888,N/A,9.529018,N/A,N/A
1,AAL,13.155,N/A,77.38235,N/A,-2.330794,N/A,0.158994,N/A,9.863492,N/A,3.100953,N/A,N/A
2,AAP,57.935,N/A,51.269913,N/A,1.581498,N/A,0.406368,N/A,12.560816,N/A,1.57468,N/A,N/A
3,AAPL,271.37,N/A,34.3519,N/A,45.245083,N/A,9.156153,N/A,25.445152,N/A,18.872094,N/A,N/A
4,ABBV,211.25,N/A,89.51272,N/A,-114.189186,N/A,6.109402,N/A,14.79164,N/A,9.878653,N/A,N/A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
417,XYL,122.11,N/A,31.150509,N/A,2.590919,N/A,3.286109,N/A,16.741677,N/A,9.124816,N/A,N/A
418,YUM,162.065,N/A,29.200901,N/A,-6.128612,N/A,5.45406,N/A,19.44197,N/A,14.919867,N/A,N/A
419,ZBH,95.985,N/A,27.038029,N/A,1.477829,N/A,2.281438,N/A,10.341026,N/A,4.497211,N/A,N/A
420,ZBRA,237.23,N/A,29.00122,N/A,3.278967,N/A,2.162667,N/A,14.249376,N/A,5.412895,N/A,N/A


## Dealing With Missing Data in Our DataFrame

Our DataFrame contains some missing data because all of the metrics we require are not available through the API we're using. 

You can use pandas' `isnull` method to identify missing data:

In [29]:
rv_dataframe[rv_dataframe.isnull().any(axis=1)]

,Ticker,Price,Number of Shares to Buy,Price-to-Earnings Ratio,PE Percentile,Price-to-Book Ratio,PB Percentile,Price-to-Sales Ratio,PS Percentile,EV/EBITDA,EV/EBITDA Percentile,EV/GP,EV/GP Percentile,RV Score
0,A,121.17,N/A,26.748343,N/A,4.958667,N/A,None,N/A,17.863888,N/A,9.529018,N/A,N/A
20,ALB,196.21,N/A,None,N/A,3.164729,N/A,4.497723,N/A,40.115954,N/A,43.943274,N/A,N/A
36,APD,292.0,N/A,None,N/A,4.218678,N/A,5.32552,N/A,100.927831,N/A,22.21233,N/A,N/A
39,ARE,48.715,N/A,None,N/A,0.537023,N/A,2.798322,N/A,12.481583,N/A,11.558957,N/A,N/A
47,BAX,18.775,N/A,None,N/A,1.575348,N/A,0.862028,N/A,8.483573,N/A,4.260789,N/A,N/A
61,CAG,14.8497,N/A,None,N/A,0.870236,N/A,0.635405,N/A,8.274734,N/A,5.272198,N/A,N/A
72,CE,62.345,N/A,None,N/A,1.687143,N/A,0.731122,N/A,12.528565,N/A,9.975623,N/A,N/A
86,CNC,38.705,N/A,None,N/A,0.953913,N/A,0.108055,N/A,6.686478,N/A,0.928184,N/A,N/A
91,COTY,2.415,N/A,None,N/A,0.600896,N/A,0.365956,N/A,6.559427,N/A,1.427104,N/A,N/A
114,DOW,35.27,N/A,None,N/A,1.580268,N/A,0.635032,N/A,16.829433,N/A,18.261725,N/A,N/A


Dealing with missing data is an important topic in data science.

There are two main approaches:

* Drop missing data from the data set (pandas' `dropna` method is useful here)
* Replace missing data with a new value (pandas' `fillna` method is useful here)

In this tutorial, we will replace missing data with the average non-`NaN` data point from that column. 

Here is the code to do this:

In [30]:
for column in ['Price-to-Earnings Ratio', 'Price-to-Book Ratio','Price-to-Sales Ratio',  'EV/EBITDA','EV/GP']:
    rv_dataframe[column].fillna(rv_dataframe[column].mean(), inplace = True)

/var/folders/5w/xpk6g_79155419v0g1n09y4c0000gn/T/ipykernel_44876/2238724823.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  rv_dataframe[column].fillna(rv_dataframe[column].mean(), inplace = True)


Now, if we run the statement from earlier to print rows that contain missing data, nothing should be returned:

In [31]:
rv_dataframe[rv_dataframe.isnull().any(axis=1)]

,Ticker,Price,Number of Shares to Buy,Price-to-Earnings Ratio,PE Percentile,Price-to-Book Ratio,PB Percentile,Price-to-Sales Ratio,PS Percentile,EV/EBITDA,EV/EBITDA Percentile,EV/GP,EV/GP Percentile,RV Score
0,A,121.17,N/A,26.748343,N/A,4.958667,N/A,None,N/A,17.863888,N/A,9.529018,N/A,N/A
20,ALB,196.21,N/A,None,N/A,3.164729,N/A,4.497723,N/A,40.115954,N/A,43.943274,N/A,N/A
36,APD,292.0,N/A,None,N/A,4.218678,N/A,5.32552,N/A,100.927831,N/A,22.21233,N/A,N/A
39,ARE,48.715,N/A,None,N/A,0.537023,N/A,2.798322,N/A,12.481583,N/A,11.558957,N/A,N/A
47,BAX,18.775,N/A,None,N/A,1.575348,N/A,0.862028,N/A,8.483573,N/A,4.260789,N/A,N/A
61,CAG,14.8497,N/A,None,N/A,0.870236,N/A,0.635405,N/A,8.274734,N/A,5.272198,N/A,N/A
72,CE,62.345,N/A,None,N/A,1.687143,N/A,0.731122,N/A,12.528565,N/A,9.975623,N/A,N/A
86,CNC,38.705,N/A,None,N/A,0.953913,N/A,0.108055,N/A,6.686478,N/A,0.928184,N/A,N/A
91,COTY,2.415,N/A,None,N/A,0.600896,N/A,0.365956,N/A,6.559427,N/A,1.427104,N/A,N/A
114,DOW,35.27,N/A,None,N/A,1.580268,N/A,0.635032,N/A,16.829433,N/A,18.261725,N/A,N/A


## Calculating Value Percentiles

We now need to calculate value score percentiles for every stock in the universe. More specifically, we need to calculate percentile scores for the following metrics for every stock:

* Price-to-earnings ratio
* Price-to-book ratio
* Price-to-sales ratio
* EV/EBITDA
* EV/GP

Here's how we'll do this:

In [33]:
metrics = {
    'Price-to-Earnings Ratio': 'PE Percentile',
    'Price-to-Book Ratio': 'PB Percentile',
    'Price-to-Sales Ratio': 'PS Percentile',
    'EV/EBITDA': 'EV/EBITDA Percentile',
    'EV/GP': 'EV/GP Percentile'
}

for metric in metrics.keys():
    rv_dataframe[metric] = pd.to_numeric(rv_dataframe[metric], errors='coerce')

for row in rv_dataframe.index:
    for metric in metrics.keys():
        rv_dataframe.loc[row, metrics[metric]] = stats.percentileofscore(rv_dataframe[metric].dropna(), rv_dataframe.loc[row, metric]) / 100

for metric in metrics.values():
    print(rv_dataframe[metric])

rv_dataframe

0       0.51928
1      0.953728
2       0.88946
3      0.727506
4      0.969152
         ...   
417    0.627249
418    0.575835
419    0.524422
420    0.573265
421    0.282776
Name: PE Percentile, Length: 422, dtype: object
0      0.656398
1       0.07346
2        0.2109
3      0.983412
4      0.007109
         ...   
417    0.440758
418    0.061611
419    0.191943
420    0.526066
421    0.898104
Name: PB Percentile, Length: 422, dtype: object
0           NaN
1      0.009501
2      0.054632
3      0.895487
4      0.769596
         ...   
417    0.541568
418    0.731591
419    0.415677
420    0.396675
421    0.726841
Name: PS Percentile, Length: 422, dtype: object
0       0.64455
1      0.206161
2      0.353081
3       0.86019
4      0.495261
         ...   
417    0.597156
418    0.720379
419    0.234597
420    0.450237
421    0.440758
Name: EV/EBITDA Percentile, Length: 422, dtype: object
0      0.547393
1      0.078199
2      0.018957
3      0.902844
4      0.563981
         ...   
4

,Ticker,Price,Number of Shares to Buy,Price-to-Earnings Ratio,PE Percentile,Price-to-Book Ratio,PB Percentile,Price-to-Sales Ratio,PS Percentile,EV/EBITDA,EV/EBITDA Percentile,EV/GP,EV/GP Percentile,RV Score
0,A,121.17,N/A,26.748343,0.51928,4.958667,0.656398,NaN,NaN,17.863888,0.64455,9.529018,0.547393,N/A
1,AAL,13.155,N/A,77.382350,0.953728,-2.330794,0.07346,0.158994,0.009501,9.863492,0.206161,3.100953,0.078199,N/A
2,AAP,57.935,N/A,51.269913,0.88946,1.581498,0.2109,0.406368,0.054632,12.560816,0.353081,1.574680,0.018957,N/A
3,AAPL,271.37,N/A,34.351900,0.727506,45.245083,0.983412,9.156153,0.895487,25.445152,0.86019,18.872094,0.902844,N/A
4,ABBV,211.25,N/A,89.512720,0.969152,-114.189186,0.007109,6.109402,0.769596,14.791640,0.495261,9.878653,0.563981,N/A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
417,XYL,122.11,N/A,31.150509,0.627249,2.590919,0.440758,3.286109,0.541568,16.741677,0.597156,9.124816,0.523697,N/A
418,YUM,162.065,N/A,29.200901,0.575835,-6.128612,0.061611,5.454060,0.731591,19.441970,0.720379,14.919867,0.774882,N/A
419,ZBH,95.985,N/A,27.038029,0.524422,1.477829,0.191943,2.281438,0.415677,10.341026,0.234597,4.497211,0.187204,N/A
420,ZBRA,237.23,N/A,29.001220,0.573265,3.278967,0.526066,2.162667,0.396675,14.249376,0.450237,5.412895,0.260664,N/A


## Calculating the RV Score
We'll now calculate our RV Score (which stands for Robust Value), which is the value score that we'll use to filter for stocks in this investing strategy.

The RV Score will be the arithmetic mean of the 4 percentile scores that we calculated in the last section.

To calculate arithmetic mean, we will use the mean function from Python's built-in statistics module.

In [34]:
from statistics import mean

for row in rv_dataframe.index:
    value_percentiles = []
    for metric in metrics.keys():
        value_percentiles.append(rv_dataframe.loc[row, metrics[metric]])
    rv_dataframe.loc[row, 'RV Score'] = mean(value_percentiles)
    
rv_dataframe

,Ticker,Price,Number of Shares to Buy,Price-to-Earnings Ratio,PE Percentile,Price-to-Book Ratio,PB Percentile,Price-to-Sales Ratio,PS Percentile,EV/EBITDA,EV/EBITDA Percentile,EV/GP,EV/GP Percentile,RV Score
0,A,121.17,N/A,26.748343,0.51928,4.958667,0.656398,NaN,NaN,17.863888,0.64455,9.529018,0.547393,NaN
1,AAL,13.155,N/A,77.382350,0.953728,-2.330794,0.07346,0.158994,0.009501,9.863492,0.206161,3.100953,0.078199,0.26421
2,AAP,57.935,N/A,51.269913,0.88946,1.581498,0.2109,0.406368,0.054632,12.560816,0.353081,1.574680,0.018957,0.305406
3,AAPL,271.37,N/A,34.351900,0.727506,45.245083,0.983412,9.156153,0.895487,25.445152,0.86019,18.872094,0.902844,0.873888
4,ABBV,211.25,N/A,89.512720,0.969152,-114.189186,0.007109,6.109402,0.769596,14.791640,0.495261,9.878653,0.563981,0.56102
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
417,XYL,122.11,N/A,31.150509,0.627249,2.590919,0.440758,3.286109,0.541568,16.741677,0.597156,9.124816,0.523697,0.546086
418,YUM,162.065,N/A,29.200901,0.575835,-6.128612,0.061611,5.454060,0.731591,19.441970,0.720379,14.919867,0.774882,0.57286
419,ZBH,95.985,N/A,27.038029,0.524422,1.477829,0.191943,2.281438,0.415677,10.341026,0.234597,4.497211,0.187204,0.310769
420,ZBRA,237.23,N/A,29.001220,0.573265,3.278967,0.526066,2.162667,0.396675,14.249376,0.450237,5.412895,0.260664,0.441381


## Selecting the 50 Best Value Stocks¶

As before, we can identify the 50 best value stocks in our universe by sorting the DataFrame on the RV Score column and dropping all but the top 50 entries.

In [35]:
rv_dataframe.sort_values(by = 'RV Score', inplace = True)
rv_dataframe = rv_dataframe[:50]
rv_dataframe.reset_index(drop = True, inplace = True)

## Calculating the Number of Shares to Buy
We'll use the `portfolio_input` function that we created earlier to accept our portfolio size. Then we will use similar logic in a for loop to calculate the number of shares to buy for each stock in our investment universe.

In [42]:
portfolio_input()

In [43]:
position_size = float(portfolio_size) / len(rv_dataframe.index)
for i in range(0, len(rv_dataframe['Ticker'])-1):
    rv_dataframe.loc[i, 'Number of Shares to Buy'] = math.floor(position_size / rv_dataframe['Price'][i])
rv_dataframe

,Ticker,Price,Number of Shares to Buy,Price-to-Earnings Ratio,PE Percentile,Price-to-Book Ratio,PB Percentile,Price-to-Sales Ratio,PS Percentile,EV/EBITDA,EV/EBITDA Percentile,EV/GP,EV/GP Percentile,RV Score
0,DXC,13.095,1,5.693479,0.007712,0.707991,0.094787,0.179791,0.011876,2.619777,0.007109,1.630122,0.021327,0.028562
1,KSS,14.6,1,6.134454,0.012853,0.403951,0.075829,0.105551,0.004751,6.251267,0.049763,1.200831,0.009479,0.030535
2,LNC,36.945,0,6.337050,0.017995,0.707813,0.092417,0.384416,0.049881,-24.963932,0.004739,-6.826296,0.00237,0.03348
3,HPQ,19.945,1,7.554924,0.028278,-23.886229,0.033175,0.325618,0.04038,5.792894,0.030806,2.267404,0.040284,0.034585
4,CMCSA,29.705,0,5.511132,0.002571,1.104850,0.132701,0.874986,0.171021,5.466968,0.023697,2.271641,0.045024,0.075003
5,UHS,181.83,0,7.874837,0.033419,1.526060,0.194313,0.639648,0.111639,6.154545,0.045024,2.109938,0.033175,0.083514
6,PRU,101.825,0,10.192693,0.051414,1.092215,0.127962,0.578693,0.090261,7.851124,0.123223,2.270836,0.042654,0.087103
7,LEG,12.1101,1,7.165739,0.025707,1.605475,0.218009,0.407352,0.057007,7.061988,0.07346,3.457796,0.104265,0.09569
8,HUM,201.8,0,20.508130,0.29563,1.378227,0.170616,0.187687,0.014252,4.947561,0.016588,0.936656,0.007109,0.100839
9,HRB,32.36,0,7.560748,0.030848,-4.983829,0.066351,1.082156,0.204276,6.902198,0.07109,3.963398,0.13981,0.102475


## Formatting Our Excel Output

We will be using the XlsxWriter library for Python to create nicely-formatted Excel files.

XlsxWriter is an excellent package and offers tons of customization. However, the tradeoff for this is that the library can seem very complicated to new users. Accordingly, this section will be fairly long because I want to do a good job of explaining how XlsxWriter works.

In [44]:
writer = pd.ExcelWriter('value_strategy.xlsx', engine='xlsxwriter')
rv_dataframe.to_excel(writer, sheet_name='Value Strategy', index = False)

## Creating the Formats We'll Need For Our .xlsx File
You'll recall from our first project that formats include colors, fonts, and also symbols like % and $. We'll need four main formats for our Excel document:

* String format for tickers
* \$XX.XX format for stock prices
* \$XX,XXX format for market capitalization
* Integer format for the number of shares to purchase
* Float formats with 1 decimal for each valuation metric

Since we already built some formats in past sections of this course, I've included them below for you. Run this code cell before proceeding.

In [45]:
background_color = '#0a0a23'
font_color = '#ffffff'

string_template = writer.book.add_format(
        {
            'font_color': font_color,
            'bg_color': background_color,
            'border': 1
        }
    )

dollar_template = writer.book.add_format(
        {
            'num_format':'$0.00',
            'font_color': font_color,
            'bg_color': background_color,
            'border': 1
        }
    )

integer_template = writer.book.add_format(
        {
            'num_format':'0',
            'font_color': font_color,
            'bg_color': background_color,
            'border': 1
        }
    )

float_template = writer.book.add_format(
        {
            'num_format':'0',
            'font_color': font_color,
            'bg_color': background_color,
            'border': 1
        }
    )

percent_template = writer.book.add_format(
        {
            'num_format':'0.0%',
            'font_color': font_color,
            'bg_color': background_color,
            'border': 1
        }
    )

In [46]:
column_formats = {
                    'A': ['Ticker', string_template],
                    'B': ['Price', dollar_template],
                    'C': ['Number of Shares to Buy', integer_template],
                    'D': ['Price-to-Earnings Ratio', float_template],
                    'E': ['PE Percentile', percent_template],
                    'F': ['Price-to-Book Ratio', float_template],
                    'G': ['PB Percentile',percent_template],
                    'H': ['Price-to-Sales Ratio', float_template],
                    'I': ['PS Percentile', percent_template],
                    'J': ['EV/EBITDA', float_template],
                    'K': ['EV/EBITDA Percentile', percent_template],
                    'L': ['EV/GP', float_template],
                    'M': ['EV/GP Percentile', percent_template],
                    'N': ['RV Score', percent_template]
                 }

for column in column_formats.keys():
    writer.sheets['Value Strategy'].set_column(f'{column}:{column}', 25, column_formats[column][1])
    writer.sheets['Value Strategy'].write(f'{column}1', column_formats[column][0], column_formats[column][1])

## Saving Our Excel Output
As before, saving our Excel output is very easy:

In [48]:
writer = pd.ExcelWriter('value_strategy.xlsx', engine='xlsxwriter')
rv_dataframe.to_excel(writer, sheet_name='Value Strategy', index=False)
workbook = writer.book
workbook.use_zip64()
writer.close()